In [198]:
## Import Libraries
import time
import requests
import pandas as pd
# import edgar

In [166]:
# Define target companies with ticker and category (CIK will be fetched from SEC)
TARGET_TICKERS = {
    'UPWK': {'name': 'Upwork', 'category': 'gig'},
    'FVRR': {'name': 'Fiverr', 'category': 'gig'},
    'UBER': {'name': 'Uber', 'category': 'gig'},
    'LYFT': {'name': 'Lyft', 'category': 'gig'},
    'DASH': {'name': 'DoorDash', 'category': 'gig'},
    # 'GRUB': {'name': 'Grubhub', 'category': 'gig'},
    'CART': {'name': 'Instacart', 'category': 'gig'},
    'ETSY': {'name': 'Etsy', 'category': 'gig'},
    'UDMY': {'name': 'Udemy', 'category': 'gig'},
    'HLF': {'name': 'Herbalife', 'category': 'mlm'},
    'PRI': {'name': 'Primerica', 'category': 'mlm'},
}

def fetch_sec_ticker_cik_mapping():
    """Fetch the official SEC ticker-to-CIK mapping."""
    url = "https://www.sec.gov/files/company_tickers.json"
    headers = {
        'User-Agent': 'Academic Research Project (gig-economy-capstone@umich.edu)',
        'Accept-Encoding': 'gzip, deflate'
    }
    
    response = requests.get(url, headers=headers, timeout=30)
    response.raise_for_status()
    data = response.json()
    
    # Build ticker -> CIK mapping
    ticker_to_cik = {}
    for entry in data.values():
        ticker = entry.get('ticker', '').upper()
        cik = str(entry.get('cik_str', ''))
        ticker_to_cik[ticker] = cik
    
    return ticker_to_cik

def build_target_companies():
    """Build TARGET_COMPANIES dict with CIKs fetched from SEC."""
    target_companies = {}
    
    # Fetch official SEC ticker-to-CIK mapping
    print("Downloading SEC ticker mapping...")
    ticker_to_cik = fetch_sec_ticker_cik_mapping()
    print(f"Loaded {len(ticker_to_cik)} ticker mappings from SEC")
    
    for ticker, info in TARGET_TICKERS.items():
        company_name = info['name']
        category = info['category']
        
        cik = ticker_to_cik.get(ticker.upper())
        
        if cik:
            target_companies[company_name] = {
                'cik': cik,
                'ticker': ticker,
                'category': category
            }
            print(f"Found CIK {cik} for {company_name} ({ticker})")
        else:
            print(f"Warning: Could not find CIK for {company_name} ({ticker})")
    
    return target_companies

# Build the TARGET_COMPANIES dictionary
print("Fetching CIK numbers from SEC EDGAR...")
TARGET_COMPANIES = build_target_companies()
print(f"\nSuccessfully retrieved CIKs for {len(TARGET_COMPANIES)} companies")
TARGET_COMPANIES

Fetching CIK numbers from SEC EDGAR...
Loaded 10435 ticker mappings from SEC
Found CIK 1627475 for Upwork (UPWK)
Found CIK 1762301 for Fiverr (FVRR)
Found CIK 1543151 for Uber (UBER)
Found CIK 1759509 for Lyft (LYFT)
Found CIK 1792789 for DoorDash (DASH)
Found CIK 1579091 for Instacart (CART)
Found CIK 1370637 for Etsy (ETSY)
Found CIK 1607939 for Udemy (UDMY)
Found CIK 1180262 for Herbalife (HLF)
Found CIK 1475922 for Primerica (PRI)

Successfully retrieved CIKs for 10 companies


{'Upwork': {'cik': '1627475', 'ticker': 'UPWK', 'category': 'gig'},
 'Fiverr': {'cik': '1762301', 'ticker': 'FVRR', 'category': 'gig'},
 'Uber': {'cik': '1543151', 'ticker': 'UBER', 'category': 'gig'},
 'Lyft': {'cik': '1759509', 'ticker': 'LYFT', 'category': 'gig'},
 'DoorDash': {'cik': '1792789', 'ticker': 'DASH', 'category': 'gig'},
 'Instacart': {'cik': '1579091', 'ticker': 'CART', 'category': 'gig'},
 'Etsy': {'cik': '1370637', 'ticker': 'ETSY', 'category': 'gig'},
 'Udemy': {'cik': '1607939', 'ticker': 'UDMY', 'category': 'gig'},
 'Herbalife': {'cik': '1180262', 'ticker': 'HLF', 'category': 'mlm'},
 'Primerica': {'cik': '1475922', 'ticker': 'PRI', 'category': 'mlm'}}

### SEC XBRL Financial Data Extraction
For structured financial metrics (Revenue, EBITDA, Net Income), using SEC's XBRL API for machine-readable data.

In [167]:
def get_company_financials_xbrl(cik, metrics=None):
    if metrics is None:
        metrics = [
            # Revenue — multiple GAAP tags used across eras/company types
            'Revenues',
            'RevenueFromContractWithCustomerExcludingAssessedTax',
            'ContractWithCustomerLiabilityChangeInTimeframePerformanceObligationSatisfiedRevenueRecognized', 
            'OperatingLeasesIncomeStatementMinimumLeaseRevenue', 'RevenueFromContractWithCustomerExcludingAssessedTax', 'RevenueRemainingPerformanceObligation', 'Revenues', 'BusinessCombinationProFormaInformationRevenueOfAcquireeSinceAcquisitionDateActual',
            'NetIncomeLoss',
            'OperatingIncomeLoss',
            'CostOfGoodsSold',
            'Assets',
            'Liabilities',
            'StockholdersEquity',
            # Metrics for Market Cap calculation
            'CommonStockSharesOutstanding',
            # Metrics to derive EBITDA
            'IncomeTaxExpenseBenefit',
        ]
    
    cik_padded = cik.zfill(10)
    url = f"https://data.sec.gov/api/xbrl/companyfacts/CIK{cik_padded}.json"
    
    headers = {
        'User-Agent': 'Academic Research Project (gig-economy-capstone@umich.edu)',
        'Accept-Encoding': 'gzip, deflate'
    }
    
    try:
        response = requests.get(url, headers=headers, timeout=30)
        response.raise_for_status()
        data = response.json()
        
        results = []
        facts = data.get('facts', {})
        
        # US-GAAP namespace contains most financial data
        us_gaap = facts.get('us-gaap', {})
        # available_metrics = list(us_gaap.keys())
        # print(f"Available US-GAAP metrics for CIK {cik}: {', '.join(list(us_gaap.keys()))} ...")
        
        # get all keys that include 'Revenue' to capture various revenue tags
        revenue_metrics = [key for key in us_gaap.keys() if 'Revenue' in key]
        # print(f"  Revenue-related metrics found: {', '.join(revenue_metrics)}")
        
        for metric in metrics:
            if metric in us_gaap:
                units = us_gaap[metric].get('units', {})
                
                # Determine which unit type to use based on metric
                if 'USD' in units:
                    data_items = units.get('USD', [])
                elif 'shares' in units:
                    data_items = units.get('shares', [])
                elif 'number' in units:
                    data_items = units.get('number', [])
                elif 'pure' in units:
                    data_items = units.get('pure', [])
                elif 'units' in units:
                    data_items = units.get('units', [])
                else:
                    available_units = list(units.keys())
                    print(f"  Warning: Metric '{metric}' has no USD or shares units for CIK {cik}")
                    print(f"    Available units: {', '.join(available_units)}")
                    continue
                
                for item in data_items:
                    if item.get('form') in ['10-K', '10-Q']:
                        results.append({
                            'form': item.get('form'),
                            'metric': metric,
                            'value': item.get('val'),
                            'fiscal_year': item.get('fy'),
                            'fiscal_period': item.get('fp'),
                            'filed_date': item.get('filed'),
                            'start_date': item.get('start'),
                            'end_date': item.get('end'),
                            'accession_number': item.get('accn')
                        })
        
        df = pd.DataFrame(results)
        
        if not df.empty:
            # Pivot to get metrics as columns
            df_pivot = df.pivot_table(
                index=['filed_date', 'fiscal_year', 'form', 'fiscal_period'],
                columns='metric',
                values='value',
                aggfunc='first'
            ).reset_index()
            
            # company, ticker, category as first columns
            df_pivot.insert(4, 'company', next((name for name, info in TARGET_COMPANIES.items() if info['cik'] == cik), 'Unknown'))
            df_pivot.insert(5, 'ticker', next((info['ticker'] for name, info in TARGET_COMPANIES.items() if info['cik'] == cik), 'Unknown'))
            df_pivot.insert(6, 'category', next((info['category'] for name, info in TARGET_COMPANIES.items() if info['cik'] == cik), 'Unknown'))
            
            # DEI fallback for CommonStockSharesOutstanding
            # DEI entries rarely have fy/fp so they can't go through the pivot;
            # instead build a filed_date → shares lookup and fill gaps directly.
            dei = facts.get('dei', {})
            dei_shares_items = dei.get('EntityCommonStockSharesOutstanding', {}).get('units', {}).get('shares', [])
            dei_lookup = {
                item['filed']: item['val']
                for item in dei_shares_items
                if item.get('form') in ['10-K', '10-Q'] and item.get('filed')
            }
            if dei_lookup:
                if 'CommonStockSharesOutstanding' not in df_pivot.columns:
                    df_pivot['CommonStockSharesOutstanding'] = pd.NA
                missing = df_pivot['CommonStockSharesOutstanding'].isna()
                df_pivot.loc[missing, 'CommonStockSharesOutstanding'] = (
                    df_pivot.loc[missing, 'filed_date'].map(dei_lookup)
                )
            
            return df_pivot
        
        return df
        
    except requests.exceptions.RequestException as e:
        print(f"Error fetching XBRL data for CIK {cik}: {e}")
        return pd.DataFrame()


def get_all_companies_financials(companies=None):
    if companies is None:
        companies = TARGET_COMPANIES
    
    all_financials = []
    
    for company_name, company_info in companies.items():
        print(f"Fetching financials for {company_name}...")
        time.sleep(0.5)  # Rate limiting
        
        try:
            df = get_company_financials_xbrl(company_info['cik'])
            if not df.empty:
                df['company'] = company_name
                df['ticker'] = company_info['ticker']
                df['category'] = company_info['category']
                all_financials.append(df)
                print(f"  Found {len(df)} annual records")
            else:
                print(f"  No financial data found")
        except Exception as e:
            print(f"  Error: {e}")
            continue
    
    if all_financials:
        combined = pd.concat(all_financials, ignore_index=True)
        combined = combined.rename_axis(None, axis=1)
        return combined
    else:
        return pd.DataFrame()


In [144]:
upwk_financials = get_company_financials_xbrl(TARGET_COMPANIES['Uber']['cik'])
nan_summary = upwk_financials.isna().sum()
print("\nNaN Summary:")
print(nan_summary)
upwk_financials.head()

  Revenue-related metrics found: ContractWithCustomerLiabilityChangeInTimeframePerformanceObligationSatisfiedRevenueRecognized, OperatingLeasesIncomeStatementMinimumLeaseRevenue, RevenueFromContractWithCustomerExcludingAssessedTax, RevenueRemainingPerformanceObligation, Revenues, BusinessCombinationProFormaInformationRevenueOfAcquireeSinceAcquisitionDateActual

NaN Summary:
metric
filed_date                                                                                        0
fiscal_year                                                                                       0
form                                                                                              0
fiscal_period                                                                                     0
company                                                                                           0
ticker                                                                                            0
category        

metric,filed_date,fiscal_year,form,fiscal_period,company,ticker,category,Assets,BusinessCombinationProFormaInformationRevenueOfAcquireeSinceAcquisitionDateActual,CommonStockSharesOutstanding,ContractWithCustomerLiabilityChangeInTimeframePerformanceObligationSatisfiedRevenueRecognized,IncomeTaxExpenseBenefit,Liabilities,NetIncomeLoss,OperatingIncomeLoss,OperatingLeasesIncomeStatementMinimumLeaseRevenue,RevenueFromContractWithCustomerExcludingAssessedTax,RevenueRemainingPerformanceObligation,Revenues,StockholdersEquity
0,2019-06-04,2019,10-Q,Q1,Uber,UBER,gig,2.398800e+10,NaN,4.571890e+08,NaN,576000000.0,1.719600e+10,3.748000e+09,-4.780000e+08,NaN,NaN,126000000.0,2.584000e+09,-8.557000e+09
1,2019-08-09,2019,10-Q,Q2,Uber,UBER,gig,2.398800e+10,NaN,4.571890e+08,NaN,604000000.0,1.719600e+10,3.748000e+09,-1.217000e+09,NaN,NaN,113000000.0,5.352000e+09,-8.557000e+09
2,2019-11-05,2019,10-Q,Q3,Uber,UBER,gig,2.398800e+10,NaN,4.571890e+08,NaN,605000000.0,1.719600e+10,1.884000e+09,-4.780000e+08,NaN,2.584000e+09,100000000.0,8.296000e+09,-7.385000e+09
3,2020-03-02,2019,10-K,FY,Uber,UBER,gig,2.398800e+10,NaN,4.571890e+08,52000000.0,-542000000.0,1.719600e+10,-4.033000e+09,-4.080000e+09,345000000.0,NaN,87000000.0,7.932000e+09,-7.385000e+09
4,2020-05-08,2020,10-Q,Q1,Uber,UBER,gig,3.176100e+10,NaN,1.716681e+09,NaN,19000000.0,1.657800e+10,-1.012000e+09,-1.034000e+09,NaN,NaN,74000000.0,3.099000e+09,1.419000e+10


In [202]:
company_financials = get_all_companies_financials(TARGET_COMPANIES)
company_financials.to_csv("../../data/sec_10K-10Q/company_financials_raw.csv", index=False)

company_financials

Fetching financials for Upwork...
  Found 30 annual records
Fetching financials for Fiverr...
  No financial data found
Fetching financials for Uber...
  Found 28 annual records
Fetching financials for Lyft...
  Found 28 annual records
Fetching financials for DoorDash...
  Found 21 annual records
Fetching financials for Instacart...
  Found 10 annual records
Fetching financials for Etsy...
  Found 44 annual records
Fetching financials for Udemy...
  Found 18 annual records
Fetching financials for Herbalife...
  Found 63 annual records
Fetching financials for Primerica...
  Found 59 annual records


,filed_date,fiscal_year,form,fiscal_period,company,ticker,category,Assets,CommonStockSharesOutstanding,IncomeTaxExpenseBenefit,Liabilities,NetIncomeLoss,OperatingIncomeLoss,RevenueFromContractWithCustomerExcludingAssessedTax,RevenueRemainingPerformanceObligation,StockholdersEquity,BusinessCombinationProFormaInformationRevenueOfAcquireeSinceAcquisitionDateActual,ContractWithCustomerLiabilityChangeInTimeframePerformanceObligationSatisfiedRevenueRecognized,OperatingLeasesIncomeStatementMinimumLeaseRevenue,Revenues
0,2018-11-08,2018,10-Q,Q3,Upwork,UPWK,gig,2.751890e+08,33740323.0,56000.0,1.400700e+08,1068000.0,1828000.0,147793000.0,NaN,-3.136700e+07,NaN,NaN,NaN,NaN
1,2019-03-07,2018,10-K,FY,Upwork,UPWK,gig,2.751890e+08,33740323.0,-1000.0,1.400700e+08,-16233000.0,-14468000.0,164445000.0,NaN,-2.143400e+07,NaN,NaN,NaN,NaN
2,2019-05-08,2019,10-Q,Q1,Upwork,UPWK,gig,3.915730e+08,106454321.0,-3000.0,1.478280e+08,-6784000.0,-6009000.0,59218000.0,NaN,-3.136700e+07,NaN,NaN,NaN,NaN
3,2019-08-07,2019,10-Q,Q2,Upwork,UPWK,gig,3.915730e+08,106454321.0,9000.0,1.478280e+08,-7196000.0,-5680000.0,121899000.0,NaN,-3.136700e+07,NaN,NaN,NaN,NaN
4,2019-11-06,2019,10-Q,Q3,Upwork,UPWK,gig,3.915730e+08,106454321.0,9000.0,1.478280e+08,-14542000.0,-9014000.0,186012000.0,NaN,-3.136700e+07,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
296,2025-02-28,2024,10-K,FY,Primerica,PRI,mlm,1.502773e+10,39368000.0,163942000.0,1.296176e+10,472068000.0,NaN,NaN,NaN,2.031254e+09,NaN,NaN,NaN,2.657451e+09
297,2025-05-08,2025,10-Q,Q1,Primerica,PRI,mlm,1.458202e+10,34996000.0,44975000.0,1.232298e+10,NaN,NaN,NaN,NaN,2.185908e+09,NaN,NaN,NaN,7.359500e+08
298,2025-08-07,2025,10-Q,Q2,Primerica,PRI,mlm,1.458202e+10,34996000.0,108442000.0,1.232298e+10,NaN,NaN,NaN,NaN,2.121760e+09,NaN,NaN,NaN,1.526904e+09
299,2025-11-06,2025,10-Q,Q3,Primerica,PRI,mlm,1.458202e+10,34996000.0,168283000.0,1.232298e+10,NaN,NaN,NaN,NaN,1.946828e+09,NaN,NaN,NaN,2.301033e+09


In [203]:
print("NaN Summary after cleaning:")
print(company_financials.isna().sum())

NaN Summary after cleaning:
filed_date                                                                                         0
fiscal_year                                                                                        0
form                                                                                               0
fiscal_period                                                                                      0
company                                                                                            0
ticker                                                                                             0
category                                                                                           0
Assets                                                                                             0
CommonStockSharesOutstanding                                                                      43
IncomeTaxExpenseBenefit                                        

In [204]:
from functools import reduce

# ── Missing Data Handling ──────────────────────────────────────────────────────

# 1. Coalesce all revenue tags → single Revenue column
rev_tags = [
    'Revenues',
    'RevenueFromContractWithCustomerExcludingAssessedTax',
    'ContractWithCustomerLiabilityChangeInTimeframePerformanceObligationSatisfiedRevenueRecognized', 
    'OperatingLeasesIncomeStatementMinimumLeaseRevenue', 'RevenueFromContractWithCustomerExcludingAssessedTax', 'RevenueRemainingPerformanceObligation', 'Revenues', 'BusinessCombinationProFormaInformationRevenueOfAcquireeSinceAcquisitionDateActual',
]
available_rev = [company_financials[c] for c in rev_tags if c in company_financials.columns]
if available_rev:
    company_financials['Revenue'] = reduce(lambda a, b: a.combine_first(b), available_rev)

# Remove original revenue columns to avoid confusion
company_financials = company_financials.drop(columns=[c for c in rev_tags if c in company_financials.columns])

# 2. Forward-fill (then back-fill) CommonStockSharesOutstanding within each company
#    DEI fallback filling is now handled inside get_company_financials_xbrl per filing;
#    this pass covers any remaining gaps across filings for the same company.
company_financials = company_financials.sort_values(['company', 'fiscal_year', 'filed_date']).reset_index(drop=True)
company_financials['CommonStockSharesOutstanding'] = (
    company_financials.groupby('company')['CommonStockSharesOutstanding']
    .transform(lambda x: x.ffill().bfill())
)

# fill any remaining gaps in key financial metrics using forward/backward fill within each company, then within each category
metrics_to_fill = ['Revenue', 'NetIncomeLoss', 'OperatingIncomeLoss', 'Assets', 'Liabilities', 'StockholdersEquity', 'IncomeTaxExpenseBenefit', 'CommonStockSharesOutstanding']
for metric in metrics_to_fill:
    if metric in company_financials.columns:
        company_financials[metric] = (
            company_financials.groupby('company')[metric]
            .transform(lambda x: x.ffill().bfill())
        )
        # If there are still missing values, try filling with average within category
        if company_financials[metric].isna().sum() > 0:
            company_financials[metric] = (
                company_financials.groupby('category')[metric]
                .transform(lambda x: x.fillna(x.mean()))
            )


print("NaN Summary after cleaning:")
print(company_financials.isna().sum())

NaN Summary after cleaning:
filed_date                      0
fiscal_year                     0
form                            0
fiscal_period                   0
company                         0
ticker                          0
category                        0
Assets                          0
CommonStockSharesOutstanding    0
IncomeTaxExpenseBenefit         0
Liabilities                     0
NetIncomeLoss                   0
OperatingIncomeLoss             0
StockholdersEquity              0
Revenue                         0
dtype: int64


In [206]:
# df for all companies that have OperatingIncomeLoss that are NaN
# df_nan_operating_income = company_financials[company_financials['CommonStockSharesOutstanding'].isna()]
# df_nan_operating_income

# function to add 1 period lag based on 'fiscal_period'
def add_lagged_metrics(df, metric, lag=1):
    # Map for fiscal period ordering
    period_order = {'Q1': 1, 'Q2': 2, 'Q3': 3, 'FY': 4}
    df['fiscal_period_order'] = df['fiscal_period'].map(period_order)
    
    df = df.sort_values(['company', 'fiscal_year', 'fiscal_period_order']).reset_index(drop=True)
    lagged_metric_name = f"{metric}_lag{lag}"
    df[lagged_metric_name] = df.groupby('company')[metric].shift(lag)
    # drop the temporary fiscal_period_order column
    df = df.drop(columns=['fiscal_period_order'])
    return df

# Add lagged metrics for Revenue, NetIncomeLoss, and OperatingIncomeLoss
for metric in ['Assets', 'CommonStockSharesOutstanding', 'IncomeTaxExpenseBenefit', 'Liabilities', 'NetIncomeLoss', 'OperatingIncomeLoss', 'StockholdersEquity', 'Revenue']:
    if metric in company_financials.columns:
        company_financials = add_lagged_metrics(company_financials, metric)
        
# Break into seperate 10K and 10Q datasets for easier analysis
company_financials_10K_only = company_financials[company_financials['form'] == '10-K'].reset_index(drop=True)

# Save cleaned dataset
company_financials.to_csv("../../data/sec_10K-10Q/company_financials_10K-10Q.csv", index=False)
company_financials_10K_only.to_csv("../../data/sec_10K-10Q/company_financials_10K-only.csv", index=False)
        
company_financials

,filed_date,fiscal_year,form,fiscal_period,company,ticker,category,Assets,CommonStockSharesOutstanding,IncomeTaxExpenseBenefit,...,StockholdersEquity,Revenue,Assets_lag1,CommonStockSharesOutstanding_lag1,IncomeTaxExpenseBenefit_lag1,Liabilities_lag1,NetIncomeLoss_lag1,OperatingIncomeLoss_lag1,StockholdersEquity_lag1,Revenue_lag1
0,2021-03-05,2020,10-K,FY,DoorDash,DASH,gig,1.732000e+09,3.999909e+08,0.0,...,-1.980000e+08,2.910000e+08,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2021-05-14,2021,10-Q,Q1,DoorDash,DASH,gig,6.353000e+09,3.999909e+08,1000000.0,...,-1.082000e+09,3.620000e+08,1.732000e+09,3.999909e+08,0.0,5.500000e+08,-204000000.0,-210000000.0,-1.980000e+08,2.910000e+08
2,2021-08-13,2021,10-Q,Q2,DoorDash,DASH,gig,6.353000e+09,3.999909e+08,1000000.0,...,-1.082000e+09,1.037000e+09,6.353000e+09,3.999909e+08,1000000.0,1.653000e+09,-129000000.0,-123000000.0,-1.082000e+09,3.620000e+08
3,2021-11-10,2021,10-Q,Q3,DoorDash,DASH,gig,6.353000e+09,3.999909e+08,2000000.0,...,-1.082000e+09,1.916000e+09,6.353000e+09,3.999909e+08,1000000.0,1.653000e+09,-129000000.0,-96000000.0,-1.082000e+09,1.037000e+09
4,2022-03-01,2021,10-K,FY,DoorDash,DASH,gig,6.353000e+09,3.999909e+08,1000000.0,...,-4.360000e+08,8.850000e+08,6.353000e+09,3.999909e+08,2000000.0,1.653000e+09,-129000000.0,-131000000.0,-1.082000e+09,1.916000e+09
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
296,2025-02-13,2024,10-K,FY,Upwork,UPWK,gig,1.037541e+09,1.372728e+08,536000.0,...,2.595170e+08,6.183180e+08,1.037541e+09,1.372728e+08,3547000.0,6.564660e+08,29513000.0,-19688000.0,2.488790e+08,5.052020e+08
297,2025-05-05,2025,10-Q,Q1,Upwork,UPWK,gig,1.211613e+09,1.353485e+08,1329000.0,...,3.810750e+08,1.909370e+08,1.037541e+09,1.372728e+08,536000.0,6.564660e+08,-89885000.0,-92624000.0,2.595170e+08,6.183180e+08
298,2025-08-06,2025,10-Q,Q2,Upwork,UPWK,gig,1.211613e+09,1.353485e+08,2510000.0,...,3.810750e+08,3.840660e+08,1.211613e+09,1.353485e+08,1329000.0,6.362360e+08,18442000.0,13049000.0,3.810750e+08,1.909370e+08
299,2025-11-04,2025,10-Q,Q3,Upwork,UPWK,gig,1.211613e+09,1.353485e+08,3636000.0,...,3.810750e+08,5.778420e+08,1.211613e+09,1.353485e+08,2510000.0,6.362360e+08,40662000.0,30830000.0,3.810750e+08,3.840660e+08


In [ ]:
# Column info helper – load once, query anytime
_col_map = pd.read_csv('../../data/sec_10K-10Q/sec_column_mapping.csv').set_index('id')

def column_info(col=None):
    """
    Print metadata for a SEC company financials dataset column.
    Call with no argument to list all available columns.
    Example: column_info('Revenue')
    """
    if col is None:
        print("Available columns:\n")
        for id_, row in _col_map.iterrows():
            print(f"  {id_:<42} {row['title']}")
        return
    if col not in _col_map.index:
        print(f"Column '{col}' not found. Call column_info() with no args to see all columns.")
        return
    row = _col_map.loc[col]
    print(f"{'Column':<22} {col}")
    print(f"{'Title':<22} {row['title']}")
    print(f"{'Units':<22} {row['units']}")
    print(f"{'Frequency':<22} {row['frequency']}")
    print(f"{'Observation range':<22} {row['observation_start']}  →  {row['observation_end']}")
    print(f"{'Seasonal adj.':<22} {row['seasonal_adjustment']}")
    print(f"{'Last updated':<22} {row['last_updated']}")
    print(f"{'Variable name':<22} {row['variable_name']}")
    print(f"\nNotes:\n  {row['notes']}")

# --- Examples ---
# column_info()              # list all columns
# column_info('Revenue')     # details for Revenue
# column_info('fiscal_period')
column_info()
